In [ ]:
import pickle
import torch
from binn import BINN, Network
import pandas as pd
import numpy as np
import torch
import random
from binn.explain import SHAPExplainer
import gc
test_input_data = pd.read_csv("/data/TCGA_primary_tpm_100.csv")
test_input_sign = pd.read_csv("/data/subtype_primary_tpm_100.csv")

model_path = f'/model/binn_model_tpm_0.001_32.pth'
model = torch.load(model_path, map_location="cuda:0",weights_only=False)
        
      
shap = SHAPExplainer(
            input_data=test_input_data, 
            design_matrix=test_input_sign, 
            model=model,
            device="cuda:0"
        )

shap.explain_cell(
            output_dir="/model", 
            iteration=0

        )

In [ ]:
import pickle
import torch
from binn import BINN, Network
import pandas as pd
import numpy as np
import torch
import random
from binn.explain import SHAPExplainer
test_input_data = pd.read_csv("/data/TCGA_primary_tpm_100.csv")
test_input_sign = pd.read_csv("/data/subtype_primary_tpm_100.csv")
import gc
model_path = f'/model/binn_model_tpm_0.001_32.pth'
model = torch.load(model_path, map_location="cuda:1",weights_only=False)
        
      
shap = SHAPExplainer(
            input_data=test_input_data, 
            design_matrix=test_input_sign, 
            model=model,
            device="cuda:1"
        )

shap.explain(
            output_dir="/model", 
            iteration=0
   
        )

In [ ]:
import pandas as pd
import numpy as np
import torch
from docs.util_for_examples import fit_data_matrix_to_network_input
from binn.explainer import BINNExplainer
from binn.based_cell_train import based_cell_train
import pickle

def _data_pre(
        data_matrix: pd.DataFrame,
        design_matrix: pd.DataFrame
):
    dfs = []
    sample_names = []  
    group_samples = design_matrix["sample"].values
    df_group = data_matrix[group_samples].T 
    dfs.append(df_group)
    sample_names.extend(group_samples)  
    X = pd.concat(dfs).fillna(0).to_numpy()
    return X,  sample_names 

def _fit_data_matrix_to_network_input(
         data_matrix: pd.DataFrame, features, feature_column="Gene"
    ) -> pd.DataFrame:
        nr_features_in_matrix = len(data_matrix.index)
        if len(features) > nr_features_in_matrix:
            features_df = pd.DataFrame(features, columns=[feature_column])
            data_matrix = data_matrix.merge(features_df, how="right", on=feature_column)
        if len(features) > 0:
            data_matrix.set_index(feature_column, inplace=True)
            data_matrix = data_matrix.loc[features]
        return data_matrix

test_input_data = pd.read_csv("/home/zxl/hdd/cellfate/data/TCGA_primary_tpm_441.csv")
test_input_sign = pd.read_csv("/home/zxl/hdd/cellfate/data/subtype_primary_tpm_441.csv",sep=',')
target_key = "Teff_CD8_cell_Tcm_layers"

with open('/home/zxl/hdd/cellfate/data/Gene_and_network.pkl', 'rb') as file:
    data = pickle.load(file)
gene_list = data["gene_list"]

input_data = test_input_data
design_matrix = test_input_sign

fitted_input_data = _fit_data_matrix_to_network_input(
                input_data.reset_index(),
                features=data["gene_list"],
                feature_column="Gene"
            )

X, y= _data_pre(
                data_matrix = fitted_input_data, design_matrix=design_matrix
            )
        
background_data = torch.Tensor(X)
test_data = torch.Tensor(X)

def get_connectivity_matrices_list():
    connectivity_matrices_list = model.B_cell_connectivity_matrices[:4]+ model.CD8_cell_Tcm_connectivity_matrices[:4]+ model.CD8_cell_Tem_connectivity_matrices[:4]+ model.CD4_cell_Th1_connectivity_matrices[:4]+model.CD4_cell_Th2_connectivity_matrices[:4]+ model.CD4_cell_Th17_connectivity_matrices[:4]+model.CD4_cell_Tfh_connectivity_matrices[:4]+ model.CD4_cell_Treg_connectivity_matrices[:4]+ model.B_cell_connectivity_matrices[6:10]+model.B_cell_connectivity_matrices[12:16]+ model.B_cell_connectivity_matrices[18:22]+model.B_cell_connectivity_matrices[24:28]+model.B_cell_connectivity_matrices[30:34]+model.CD8_cell_Tcm_connectivity_matrices[6:10]+model.CD8_cell_Tcm_connectivity_matrices[12:16]+  model.CD8_cell_Tcm_connectivity_matrices[18:22]+ model.CD8_cell_Tcm_connectivity_matrices[24:28]+ model.CD8_cell_Tcm_connectivity_matrices[30:34]+model.CD8_cell_Tem_connectivity_matrices[6:10]+  model.CD8_cell_Tem_connectivity_matrices[12:16]+  model.CD8_cell_Tem_connectivity_matrices[18:22]+ model.CD8_cell_Tem_connectivity_matrices[24:28]+ model.CD8_cell_Tem_connectivity_matrices[30:34]+model.CD4_cell_Th1_connectivity_matrices[6:10]+ model.CD4_cell_Th1_connectivity_matrices[12:16]+ model.CD4_cell_Th1_connectivity_matrices[18:22]+ model.CD4_cell_Th1_connectivity_matrices[24:28]+ model.CD4_cell_Th2_connectivity_matrices[6:10]+ model.CD4_cell_Th2_connectivity_matrices[12:16]+ model.CD4_cell_Th2_connectivity_matrices[18:22]+ model.CD4_cell_Th2_connectivity_matrices[24:28]+model.CD4_cell_Th17_connectivity_matrices[6:10]+ model.CD4_cell_Th17_connectivity_matrices[12:16]+ model.CD4_cell_Th17_connectivity_matrices[18:22]+ model.CD4_cell_Th17_connectivity_matrices[24:28]+model.CD4_cell_Tfh_connectivity_matrices[6:10]+ model.CD4_cell_Tfh_connectivity_matrices[12:16]+ model.CD4_cell_Tfh_connectivity_matrices[18:22]+ model.CD4_cell_Tfh_connectivity_matrices[24:28]+model.CD4_cell_Treg_connectivity_matrices[6:10]+ model.CD4_cell_Treg_connectivity_matrices[12:16]+ model.CD4_cell_Treg_connectivity_matrices[18:22]+ model.CD4_cell_Treg_connectivity_matrices[24:28]
    return connectivity_matrices_list
connectivity_matrices_list = get_connectivity_matrices_list()
filename="/model/binn_model0_tpm_0.001_32.pth"
model = torch.load(filename, map_location="cuda:1", weights_only=False)

explainer =BINNExplainer(model)
shap_dict = explainer._explain_cell_layer(
            test_data, y, background_data
            )
df = explainer.shap_single_go(shap_dict,y,connectivity_matrices_list,target_key)


In [ ]:
import pandas as pd
import numpy as np
import torch
import random
import os
from docs.util_for_examples import fit_data_matrix_to_network_input
from binn.explainer import BINNExplainer
from binn.based_cell_train import based_cell_train
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, train_test_split
import pickle

test_input_data = pd.read_csv("/home/zxl/hdd/cellfate/data/TCGA_primary_tpm_441.csv")
test_input_sign = pd.read_csv("/home/zxl/hdd/cellfate/data/subtype_primary_tpm_441.csv",sep=',')

with open('/home/zxl/hdd/cellfate/data/Gene_and_network1.pkl', 'rb') as file:
    data = pickle.load(file)
gene_list = data["gene_list"]

gene_set = set(gene_list.tolist()) 
input_data = test_input_data
design_matrix = test_input_sign

fitted_input_data = _fit_data_matrix_to_network_input(
                    input_data.reset_index(),
                    features=data["gene_list"],
                    feature_column="Gene"
                )

X, y = _data_pre(fitted_input_data, design_matrix=design_matrix)

test_data = torch.Tensor(X)
background_data = torch.Tensor(X)

filename="/home/zxl/hdd/cellfate/model4/binn_model_tpm_0.001_32.pth"
model = torch.load(filename, map_location="cuda:1", weights_only=False)
explainer =BINNExplainer(model)
        
shap_dict = explainer._explain_cell_layer(
                            test_data, y, background_data
            )

df = explainer.shap_single_cell(shap_dict,y)    


